# Section A: Data Preparation
===========================

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('studentsMark.csv')

# Cleaning: Drop the unique identifier as it doesn't help with prediction
df_cleaned = df.drop(columns=['StudenttID'])

# Quality Check: Convert categorical 'Gender' to numerical values (Female=0, Male=1)
# This ensures all data is in a format suitable for mathematical models
df_cleaned['Gender'] = df_cleaned['Gender'].map({'Female': 0, 'Male': 1})

# Separation: Separate the feature columns (X) from the target column (y)
X = df_cleaned.drop(columns=['Result'])

y = df_cleaned['Result']

# Display the first few rows of the processed features and target
print("Feature columns (X):")
print(X.head())
print("\nTarget column (y):")
print(y.head())

Feature columns (X):
   Gender  attend class  study
0       0             0      1
1       1             1      0
2       0             1      1
3       1             0      0
4       0             1      0

Target column (y):
0    1
1    0
2    1
3    0
4    0
Name: Result, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split

# Split the data: 65% for training and 35% for testing
# random_state ensures that the split is reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35, random_state=42)

# Verify the sizes of the split datasets
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Training set size: 32 samples
Test set size: 18 samples


===========================
# Section B: Model Building and Evaluation
===========================

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Initialize the Logistic Regression model
log_reg = LogisticRegression(random_state=42)

# Define the hyperparameter grid to search
# We are tuning 'C' (regularization strength) and 'solver' (optimization algorithm)
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

# Use GridSearchCV to find the best combination of hyperparameters
# cv=5 means 5-fold cross-validation
grid_search = GridSearchCV(estimator=log_reg, param_grid=param_grid, cv=5)

# Train the model using the training set
grid_search.fit(X_train, y_train)

# Extract the best model and parameters
best_model = grid_search.best_estimator_
print(f"Best Hyperparameters found: {grid_search.best_params_}")

# Display training accuracy of the tuned model
train_accuracy = best_model.score(X_train, y_train)
print(f"Training Accuracy with tuned model: {train_accuracy * 100:.2f}%")

Best Hyperparameters found: {'C': 1, 'solver': 'liblinear'}
Training Accuracy with tuned model: 100.00%


In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# Initialize the Decision Tree Classifier
dt_clf = DecisionTreeClassifier(random_state=42)

# Define the hyperparameter grid
# We are tuning 'max_depth' and 'min_samples_split'
dt_param_grid = {
    'max_depth': [None, 2, 4, 6, 8, 10],
    'min_samples_split': [2, 5, 10]
}

# Use GridSearchCV for hyperparameter tuning
dt_grid_search = GridSearchCV(estimator=dt_clf, param_grid=dt_param_grid, cv=5)

# Train the model using the training set
dt_grid_search.fit(X_train, y_train)

# Extract the best model and parameters
best_dt_model = dt_grid_search.best_estimator_
print(f"Best Decision Tree Hyperparameters: {dt_grid_search.best_params_}")

# Display training accuracy
dt_train_accuracy = best_dt_model.score(X_train, y_train)
print(f"Decision Tree Training Accuracy: {dt_train_accuracy * 100:.2f}%")

Best Decision Tree Hyperparameters: {'max_depth': None, 'min_samples_split': 2}
Decision Tree Training Accuracy: 100.00%


In [6]:
from sklearn.metrics import recall_score, f1_score, classification_report

# 1. Generate predictions for the test set using both models
y_pred_lr = best_model.predict(X_test)
y_pred_dt = best_dt_model.predict(X_test)

# 2. Calculate Recall and F1-score for Logistic Regression
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)

# 3. Calculate Recall and F1-score for Decision Tree
dt_recall = recall_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)

# 4. Print the results
print("--- Logistic Regression Performance ---")
print(f"Recall: {lr_recall:.2f}")
print(f"F1-Score: {lr_f1:.2f}")

print("\n--- Decision Tree Performance ---")
print(f"Recall: {dt_recall:.2f}")
print(f"F1-Score: {dt_f1:.2f}")

# Optional: Full classification report for detailed analysis
print("\nFull Logistic Regression Report:")
print(classification_report(y_test, y_pred_lr))

--- Logistic Regression Performance ---
Recall: 1.00
F1-Score: 1.00

--- Decision Tree Performance ---
Recall: 1.00
F1-Score: 1.00

Full Logistic Regression Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         8

    accuracy                           1.00        18
   macro avg       1.00      1.00      1.00        18
weighted avg       1.00      1.00      1.00        18



# Model Comparison and Selection

### 1. Performance Comparison
Both models achieved perfect scores on the test set, indicating that the features provided (`Gender`, `attend class`, `study`) are strong predictors of the `Result`.

| Metric | Logistic Regression | Decision Tree |
| :--- | :---: | :---: |
| **Recall** | $1.00$ | $1.00$ |
| **F1-Score** | $1.00$ | $1.00$ |
| **Accuracy** | $1.00$ | $1.00$ |

### 2. Justification of Results
The identical metric values ($1.00$) suggest that the dataset has a very clear linear or hierarchical separation. In this specific case, students who studied and attended class consistently passed, making it easy for both algorithms to define a perfect decision boundary.

### 3. Which Model is Better?
While both models performed identically in terms of metrics, **Logistic Regression** is considered more suitable for this specific case.

#### Why Logistic Regression?
* **Robustness on Small Data:** With only 50 samples, Decision Trees are highly susceptible to overfitting (memorizing the noise in the data). Logistic Regression is a simpler model that generalizes better on small datasets.
* **Probabilistic Insights:** Logistic Regression calculates the probability of a result. For an educator, knowing a student has a 51% chance of passing vs. a 99% chance is more useful than a simple "Pass/Fail" label.
* **Efficiency:** It is computationally efficient and provides clear coefficients that show the direct impact of each feature (e.g., how much "studying" increases the odds of passing compared to "attendance").